# Task 2 — RAG-Based QA on TurkishMMLU History

**Önerilen runtime:** T4 16GB yeterli (~45 dk, <2 CU). A100 gerekmez.

Pipeline (sırayla):
1. Drive mount + HF cache Drive'a
2. Repo clone + pip install
3. Model prefetch (Qwen2.5-7B, multilingual-e5-base)
4. TurkishMMLU History filtreleme
5. Tarih kitabı PDF'lerinin Drive'a yüklü olduğunu doğrula
6. KB ingest: PDF → chunk → e5 embed → FAISS
7. Zero-shot accuracy (bağlamsız baseline)
8. RAG accuracy (top_k=5)
9. Sonuçları zip + lokale indir

In [ ]:
# === Drive mount + HF cache Drive'a ===
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/llm_final'
os.environ['DRIVE_ROOT']         = DRIVE_ROOT
os.environ['HF_HOME']            = f'{DRIVE_ROOT}/hf_cache'
os.environ['HF_DATASETS_CACHE']  = f'{DRIVE_ROOT}/hf_cache/datasets'

for sub in ('hf_cache', 'data/history_book', 'data/faiss_index',
            'data/turkish_mmlu_history', 'results', 'exports'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

print('Drive root:', DRIVE_ROOT)

In [ ]:
# === GPU doğrulama ===
!nvidia-smi -L
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# === Repo clone + bağımlılıklar (HW2-style: torch'a dokunma) ===
REPO_URL = 'https://github.com/mustafagalata/lora-project.git'  # << kendi repo URL'ini yaz
%cd /content
!rm -rf /content/repo && git clone {REPO_URL} /content/repo
%cd /content/repo

# Temel paketler — torch ve torchvision'a açıkça dokunmuyoruz (Colab uyumlu pair kalır)
!pip install -q \
    transformers \
    peft \
    accelerate \
    bitsandbytes \
    datasets \
    sentence-transformers \
    "faiss-cpu>=1.8.0" \
    langchain langchain-community langchain-huggingface langchain-text-splitters \
    pypdf pdfplumber tiktoken \
    pyyaml numpy pandas tqdm matplotlib

# Task 3 design (sadece tasarım dosyaları için, runtime'da kullanılmaz)
!pip install -q langdetect wikipedia-api

In [ ]:
# === Model prefetch (ilk session'da Drive'a yazılır) ===
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen2.5-7B-Instruct')         # ~15GB (Task 1'le paylaşır)
snapshot_download('intfloat/multilingual-e5-base')    # ~500MB

## 1) TurkishMMLU History filtreleme

In [ ]:
!python -m src.task2_rag.prepare_mmlu

## 2) Tarih kitap PDF'lerini doğrula

PDF'leri Colab oturumu açmadan önce `$DRIVE_ROOT/data/history_book/` altına manuel olarak yüklemelisin (Drive web arayüzünden). Aşağıdaki hücre içeriği listeler.

In [ ]:
import glob
pdfs = sorted(glob.glob(f'{DRIVE_ROOT}/data/history_book/*.pdf'))
print(f'Bulunan PDF: {len(pdfs)}')
for p in pdfs:
    sz_mb = os.path.getsize(p) / 1e6
    print(f'  {os.path.basename(p)}  ({sz_mb:.1f} MB)')
if not pdfs:
    raise SystemExit('Önce Drive/data/history_book/ altına PDF yükle.')

## 3) KB ingest: PDF → chunk → e5 embed → FAISS

PDF'lerin başındaki içindekiler/sunuş ve sonundaki kaynakça/dizin blokları otomatik atılır; çok kısa / dot-leader yoğun chunk'lar elenir. Davranış `config.yaml` içinde `task2.filter` altından kontrol edilir.

Çıktı: `$DRIVE_ROOT/data/faiss_index/turkish_history.{faiss,pkl}`

In [ ]:
!python -m src.task2_rag.ingest_kb

## 4) Zero-shot accuracy (bağlamsız baseline)

In [ ]:
!python -m src.task2_rag.eval_zero_shot

## 5) RAG accuracy

`top_k` retrieval ayarı `config.yaml` içinde `task2.top_k` (default: 5).

In [ ]:
!python -m src.task2_rag.eval_rag

## 6) Karşılaştırma özeti

In [ ]:
import json
zs  = json.load(open(f'{DRIVE_ROOT}/results/task2_zero_shot_accuracy.json'))
rag = json.load(open(f'{DRIVE_ROOT}/results/task2_rag_accuracy.json'))
print(f"Zero-shot:  {zs['accuracy']:.4f}  ({zs['correct']}/{zs['n']})")
print(f"RAG (k={rag['top_k']}): {rag['accuracy']:.4f}  ({rag['correct']}/{rag['n']})")
print(f"Δ = {(rag['accuracy'] - zs['accuracy']) * 100:+.2f} pp")

## 7) Sonuçları zip + lokale indir

In [ ]:
import os, shutil
out_dir = f'{DRIVE_ROOT}/exports/task2_artifacts'
shutil.rmtree(out_dir, ignore_errors=True)
os.makedirs(out_dir, exist_ok=True)
shutil.copytree(f'{DRIVE_ROOT}/results', f'{out_dir}/results')
shutil.copytree(f'{DRIVE_ROOT}/data/faiss_index', f'{out_dir}/faiss_index')
shutil.make_archive(out_dir, 'zip', out_dir)
print('Wrote:', out_dir + '.zip')